# CineMetrix

## Phase 1: Data Understanding

Objective:
Understand the dataset before analysis and identify potential data quality issues.
This notebook performs read-only diagnostic exploration. No values are modified, dropped, or imputed here — all such decisions and their justifications are made explicitly in notebook 2

In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv("../data/raw/Netflix.csv")

## Data Dictionary

| Column | Expected meaning | Notes |
|---|---|---|
| show_id | Unique identifier per title | |
| type | Movie or TV Show | |
| director | Director name(s), comma-separated if multiple | High missingness expected for TV shows |
| cast | Actor names, comma-separated | |
| country | Production country, comma-separated if co-produced | Composite values — needs splitting before country-level analysis |
| date_added | Date title was added to Netflix catalog | Stored as string, needs datetime conversion |
| release_year | Year of original release | |
| rating | Content rating (TV-MA, PG-13, etc.) | |
| duration | Minutes for Movies, seasons for TV Shows | Same column, two different units — needs splitting |
| genres | Genre tags, comma-separated | Composite values — needs splitting |
| description | Free-text synopsis | Unused currently — candidate for NLP |

In [3]:
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,genres,description
0,s1,TV Show,3%,NaN,"João Miguel, Bianca Comparato, Michel Gomes, R...",Brazil,14-Aug-20,2020,TV-MA,4,"International TV Shows, TV Dramas, TV Sci-Fi &...",In a future where the elite inhabit an island ...
1,s10,Movie,1920,Vikram Bhatt,"Rajneesh Duggal, Adah Sharma, Indraneil Sengup...",India,15-Dec-17,2008,TV-MA,143,"Horror Movies, International Movies, Thrillers",An architect and his wife move into a castle t...
2,s100,Movie,3 Heroines,Iman Brotoseno,"Reza Rahadian, Bunga Citra Lestari, Tara Basro...",Indonesia,5-Jan-19,2016,TV-PG,124,"Dramas, International Movies, Sports Movies",Three Indonesian women break records by becomi...
3,s1000,Movie,Blue Mountain State: The Rise of Thadland,Lev L. Spiro,"Alan Ritchson, Darin Brooks, James Cade, Rob R...",United States,1-Mar-16,2016,R,90,Comedies,New NFL star Thad buys his old teammates' belo...
4,s1001,TV Show,Blue Planet II,NaN,David Attenborough,United Kingdom,3-Dec-18,2017,TV-G,1,"British TV Shows, Docuseries, Science & Nature TV",This sequel to the award-winning nature series...


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7787 entries, 0 to 7786
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       7787 non-null   str  
 1   type          7787 non-null   str  
 2   title         7787 non-null   str  
 3   director      5398 non-null   str  
 4   cast          7069 non-null   str  
 5   country       7280 non-null   str  
 6   date_added    7777 non-null   str  
 7   release_year  7787 non-null   int64
 8   rating        7780 non-null   str  
 9   duration      7787 non-null   int64
 10  genres        7787 non-null   str  
 11  description   7787 non-null   str  
dtypes: int64(2), str(10)
memory usage: 730.2 KB


### Observation

The column `date_added` is stored as object instead of datetime.
It needs datatype conversion before time-series analysis.

In [5]:
df.shape

(7787, 12)

### Observation

The dataset contains 7,787 Netflix titles and 12 features.
The dataset is sufficiently large for exploratory analysis and dashboard development.

In [6]:
df.describe()

,release_year,duration
count,7787.000000,7787.000000
mean,2013.932580,69.122769
std,8.757395,50.950743
min,1925.000000,1.000000
25%,2013.000000,2.000000
50%,2017.000000,88.000000
75%,2018.000000,106.000000
max,2021.000000,312.000000


### Observation

The dataset spans titles released between 1925 and 2021,
making long-term trend analysis possible.

In [7]:
df.describe(include="object")

C:\Users\HP\AppData\Local\Temp\ipykernel_14132\702825166.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include="object")


,show_id,type,title,director,cast,country,date_added,rating,genres,description
count,7787,7787,7787,5398,7069,7280,7777,7780,7787,7787
unique,7787,2,7787,4049,6831,681,1565,14,492,7769
top,s1,Movie,3%,"Raúl Campos, Jan Suter",David Attenborough,United States,1-Jan-20,TV-MA,Documentaries,Multiple women report their husbands as missin...
freq,1,5377,1,18,18,2555,118,2863,334,3


In [8]:
df.isnull().sum()

show_id            0
type               0
title              0
director        2389
cast             718
country          507
date_added        10
release_year       0
rating             7
duration           0
genres             0
description        0
dtype: int64

In [9]:
missing_percentage = (df.isnull().sum()/len(df))*100
missing_percentage.sort_values(ascending=False)

director        30.679337
cast             9.220496
country          6.510851
date_added       0.128419
rating           0.089893
title            0.000000
show_id          0.000000
type             0.000000
release_year     0.000000
duration         0.000000
genres           0.000000
description      0.000000
dtype: float64

### Observation

Approximately 30% of records have missing director information.
This column requires careful treatment before analysis.

In [10]:
df.duplicated().sum()

np.int64(0)

### Observation

No duplicate records were detected.
Therefore, duplicate removal is unnecessary.

In [11]:
df.nunique()

show_id         7787
type               2
title           7787
director        4049
cast            6831
country          681
date_added      1565
release_year      73
rating            14
duration         206
genres           492
description     7769
dtype: int64

### Observation

The variable `type` contains only two categories:
Movie and TV Show.

# Data Quality Report

### Observation

- Director has the highest missing percentage (30.68%).
- Cast and Country also contain missing values.
- Other variables are nearly complete.
- Missing values should be investigated before deciding whether to drop or impute them.

In [14]:
for col in df.columns:
    print(f"\n{col}")
    print(df[col].head(3).tolist())


show_id
['s1', 's10', 's100']

type
['TV Show', 'Movie', 'Movie']

title
['3%', '1920', '3 Heroines']

director
[nan, 'Vikram Bhatt', 'Iman Brotoseno']

cast
['João Miguel, Bianca Comparato, Michel Gomes, Rodolfo Valente, Vaneza Oliveira, Rafael Lozano, Viviane Porto, Mel Fronckowiak, Sergio Mamberti, Zezé Motta, Celso Frateschi', 'Rajneesh Duggal, Adah Sharma, Indraneil Sengupta, Anjori Alagh, Rajendranath Zutshi, Vipin Sharma, Amin Hajee, Shri Vallabh Vyas', 'Reza Rahadian, Bunga Citra Lestari, Tara Basro, Chelsea Islan']

country
['Brazil', 'India', 'Indonesia']

date_added
['14-Aug-20', '15-Dec-17', '5-Jan-19']

release_year
[2020, 2008, 2016]

rating
['TV-MA', 'TV-MA', 'TV-PG']

duration
[4, 143, 124]

genres
['International TV Shows, TV Dramas, TV Sci-Fi & Fantasy', 'Horror Movies, International Movies, Thrillers', 'Dramas, International Movies, Sports Movies']

description
['In a future where the elite inhabit an island paradise far from the crowded slums, you get one chance to

In [15]:
df.sample(10, random_state=42)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,genres,description
7324,s7591,Movie,Whindersson Nunes: Adult,Diego Pignataro,Whindersson Nunes,Brazil,15-Aug-19,2019,TV-MA,69,Stand-Up Comedy,Brazilian YouTube sensation Whindersson Nunes ...
4694,s5223,TV Show,Rita,NaN,"Mille Dinesen, Carsten Bjørnlund, Lise Baastru...",Denmark,15-Aug-20,2020,TV-MA,5,"International TV Shows, TV Comedies, TV Dramas","Independent, outspoken and adored by her stude..."
1322,s2189,Movie,FirstBorn,Nirpal Bhogal,"Antonia Thomas, Luke Norris, Thea Petrie, Eile...",United Kingdom,31-Mar-17,2016,TV-MA,90,"Horror Movies, International Movies",A young couple fights supernatural foes in an ...
5106,s5595,Movie,Si Doel the Movie 3,Rano Karno,"Rano Karno, Cornelia Agatha, Maudy Koesnaedi, ...",Indonesia,23-May-20,2020,TV-G,93,"Dramas, International Movies, Music & Musicals",Torn between reuniting with one family and lea...
101,s109,TV Show,3Below: Tales of Arcadia,NaN,"Tatiana Maslany, Diego Luna, Nick Offerman, Ni...",United States,12-Jul-19,2019,TV-Y7,2,"Kids' TV, TV Action & Adventure, TV Sci-Fi & F...","After crash-landing on Earth, two royal teen a..."
3653,s4287,Movie,Muran,Rajan Madhav,"Prasanna, Cheran, Haripriya, Nikhita, Suma Bha...",India,1-Oct-18,2011,TV-14,134,"Action & Adventure, International Movies",When jingle composer Nanda's car breaks down o...
3519,s4166,Movie,Mojave,William Monahan,"Garrett Hedlund, Oscar Isaac, Louise Bourgoin,...",United States,26-Jul-18,2015,R,93,"Dramas, Independent Movies, Thrillers",A lethal game of cat and mouse stretches from ...
487,s1437,TV Show,Club Friday To Be Continued - The Promise,NaN,"Jirayu La-ongmanee, Focus Jirakul",NaN,20-Apr-18,2016,TV-MA,1,"International TV Shows, Romantic TV Shows, TV ...",Two young lovers vow to stay friends through t...
6022,s6419,TV Show,The Great British Baking Show: The Beginnings,NaN,"Paul Hollywood, Mary Berry, Mel Giedroyc, Sue ...",NaN,1-Nov-18,2012,TV-PG,1,"British TV Shows, Reality TV",A dozen amateur bakers amiably compete for the...
3274,s3946,Movie,Marlon Wayans: Woke-ish,Marcus Raboy,Marlon Wayans,United States,27-Feb-18,2018,TV-MA,67,Stand-Up Comedy,"Rollicking, outrageous and audacious, Marlon W..."


In [16]:
df["type"].value_counts()

type
Movie      5377
TV Show    2410
Name: count, dtype: int64

In [17]:
df["type"].value_counts(normalize=True)*100

type
Movie      69.050982
TV Show    30.949018
Name: proportion, dtype: float64

### Observation

- The dataset contains 5,377 Movies and 2,410 TV Shows.
- Movies represent approximately 69% of the dataset.
- Any comparison between Movies and TV Shows should consider this imbalance.

In [18]:
df["rating"].value_counts(dropna=False)

rating
TV-MA       2863
TV-14       1931
TV-PG        806
R            665
PG-13        386
TV-Y         280
TV-Y7        271
PG           247
TV-G         194
NR            84
G             39
NaN            7
TV-Y7-FV       6
UR             5
NC-17          3
Name: count, dtype: int64

In [19]:
df["country"].value_counts().head(20)

country
United States                    2555
India                             923
United Kingdom                    397
Japan                             226
South Korea                       183
Canada                            177
Spain                             134
France                            115
Egypt                             101
Turkey                            100
Mexico                            100
Australia                          83
Taiwan                             78
Brazil                             72
Philippines                        71
Indonesia                          70
Nigeria                            70
United Kingdom, United States      64
Germany                            61
United States, Canada              60
Name: count, dtype: int64

### Observation

- Some records contain multiple countries in a single field.
- Country-level analysis may require parsing these values.

In [20]:
df["genres"].value_counts().head(20)

genres
Documentaries                                        334
Stand-Up Comedy                                      321
Dramas, International Movies                         320
Comedies, Dramas, International Movies               243
Dramas, Independent Movies, International Movies     215
Kids' TV                                             205
Children & Family Movies                             177
Documentaries, International Movies                  172
Children & Family Movies, Comedies                   169
Comedies, International Movies                       161
Dramas, International Movies, Romantic Movies        153
Comedies, International Movies, Romantic Movies      139
Dramas                                               117
Action & Adventure, Dramas, International Movies     117
International TV Shows, TV Dramas                    111
Dramas, International Movies, Thrillers              109
Crime TV Shows, International TV Shows, TV Dramas    106
Comedies, Dramas, Indepe

### Observation

- Many genre values are combinations of multiple genres.
- The column is not normalized.
- Future analysis may require splitting genres into individual categories.

In [21]:
df["duration"].describe()

count    7787.000000
mean       69.122769
std        50.950743
min         1.000000
25%         2.000000
50%        88.000000
75%       106.000000
max       312.000000
Name: duration, dtype: float64

In [22]:
df["duration"].sort_values().head(20)

58      1
55      1
54      1
52      1
4       1
7775    1
7777    1
7780    1
10      1
15      1
34      1
27      1
23      1
2014    1
2003    1
1998    1
5510    1
5507    1
7767    1
5533    1
Name: duration, dtype: int64

In [23]:
df["duration"].sort_values(ascending=False).head(20)

7741    312
6502    253
3880    237
2995    233
4567    230
4864    228
2804    224
2441    214
2538    209
6121    209
3873    208
6013    205
1286    204
7306    203
6242    201
883     200
1082    196
4925    195
6271    195
4837    194
Name: duration, dtype: int64

### Observation
The `duration` column mixes two different units depending on `type`:
movies are measured in minutes, TV shows in number of seasons.
The minimum value of 1 and the 2nd-percentile value of 2 are TV show
season counts, not short movies. Any aggregate statistic on this raw
column (mean, std, percentile) is computing across two incompatible
units and should not be interpreted directly. Resolution deferred to
Phase 2 (Data Cleaning) — split into `duration_minutes` and `seasons`.

## Data Quality Findings Summary

| Issue | Column(s) | Severity | Action required |
|---|---|---|---|
| Missing values | director (30.7%), cast (9.2%), country (6.5%) | Medium | Investigate missingness pattern (random vs systematic) before deciding |
| Wrong dtype | date_added (string, should be datetime) | High | Convert before any time-series analysis |
| Composite values | country (681 unique strings, true count lower) | High | Split and explode before country-level analysis |
| Composite values | genres (492 unique combinations) | High | Split and explode before genre-level analysis |
| Mixed units in one column | duration (minutes for movies, seasons for TV shows) | High | Split into duration_minutes and seasons |
| Class imbalance | type (69% Movie, 31% TV Show) | Low | Note when comparing groups, no fix needed |